## Adapting ANNABELL 

This implementation seeks to adapt some of ANNABELL's functionality into pyClarion's four main stages of implementation: keyspace definition, model construction, knowledge intialization, and event processing.

The features that are sought to be adapted are ANNABELL's:
- Conversational ability.
- Virtual enviroment, and its ability to move throughout it.
- Some of it's context, like its recognition of a self.

Considering ANNABELL's complexity and the ultimate requirements of this project, the capability of these features will be much smaller. For example, the text-based virtual enviroment will have less rooms than ANNABELL's implementation.

### Keyspace Definition

Intuitively, it seems like this would be the most appropriate place to adapt ANNABELL's databases, which are used to train and test the system. The keyspace exists as the system's context or world so to speak, so this lines up pretty well. Again, they will not be adapted to their full extent, and only the relevant ones to the project will be used:
- Parts of the body
- Communication interactions
- text-based environment
- people/self

In [ ]:

from pyClarion import Atom # Represents an atomic term
from pyClarion import Atoms # Represents a sort grouping atomic terms
from pyClarion.knowledge import Buses, Bus, BusFamily, DataFamily, Root


class Partsofbody(Atoms): 
    larm: Atom
    rarm: Atom
    lleg: Atom
    rleg: Atom

class Communication(Atoms): 
    yes: Atom
    no: Atom

class Environment(Atoms):
    room0: Atom
    room1: Atom
    room2: Atom
    room3: Atom
    room4: Atom

class People(Atoms):
    self: Atom
    friend: Atom
    brother: Atom

class Main(Buses):
    input: Bus
    output: Bus
    target: Bus


class Buses(BusFamily):
    main: Main


class Data(DataFamily):
    parts: Partsofbody
    comm: Communication
    env: Environment
    people: People
    


class Root(Root):
    b: Buses
    d: Data


### Model Construction

Model construction is the process of initializing and assembling a collection of `pyClarion` component processes that implement the functionality of a desired model. 

The `pyClarion` library defines component processes implementing the main elements of the Clarion cognitive architecture, and it allows models to be constructed in a modular and compositional fashion.

For the present purposes, model construction requires the `Agent`, `Input`, and `ChunkStore` processes and the `WCSTData` keyspace.

In [ ]:
"""Agent construction for bottom-up activation model."""

from pyClarion import Agent # Initialzes keyspaces specific to an agent
from pyClarion import Input # Receives activations from external sources
from pyClarion import ChunkStore # Maintains a collection of Clarion chunks
from pyClarion import BottomUp
from pyClarion.knowledge import DVPairs


class BottomUpAgent[R: Root, D: DVPairs](Agent):
    root: R
    ipt: Input
    chunks: ChunkStore[D]
    bu: BottomUp[D]

    def __init__(self, name: str, root: R, f: DataFamily, d: D) -> None:
        
        super().__init__(name, root)

        # Convenient handle for keyspace root.
        self.root = root
        
        # Any process objects initialized within this block will automatically 
        # be added to the agent's system.
        with self:
            # Create an input process to pass in feature activations. The tuple 
            # `(b, d)`indicates that this process will receive inputs in the 
            # form of dimension-value pairs constructed by pairing symbols from 
            # the families 'b'  and 'd'.
            self.ipt = Input(f"{name}.ipt", d)

            # Create a chunk store using the family 'd' to house chunk symbols 
            # (arg 'c')
            self.chunks = ChunkStore(f"{name}.chunks", c=f, d=d)

            # Create a bottom up activation process dependent on chunk store.
            self.bu = self.ipt >> self.chunks.bottom_up(f"{name}.bu")

# Initialize basic data symbols for current simulation
# During initialization, WCSTData() will automatically generate symbols 
# for all terms annotated with a sort or term type.
root = Root()

# Create a new bottom-up demo agent and populate its keyspace with data symbols.
agent = BottomUpAgent("agent", root, root.d, (root.b.main, root.d))


Note that a model may contain more processes than were explicitly specified. This is because some processes may depend on or extend the functionality of more elementary processes. 

In [4]:
# List all processes associated with the current model
agent.system.procs

[<BottomUpAgent 'agent' at 0x107b64d70>,
 <Input 'agent.ipt' at 0x107b64ec0>,
 <ChunkStore 'agent.chunks' at 0x107b65160>,
 <BottomUp 'agent.bu' at 0x107b657f0>]

### Knowledge Initialization

Knowledge initialization is the process of defining any prior knowledge available to a model. 

Typically, this involves the specification of user-defined explicit knowledge such as chunks and/or rules using `pyClarion`'s knowledge representation tools. It may also, however, include populating the model with other prior knowledge, such as pretrained neural network weights. 

For the present example, knowledge initialization involves defining some chunks for which to calculate bottom-up activations. Three chunks are defined below, representing three distinct card faces.

In [5]:
"""Chunks for bottom-up activation model"""

# Define short handles for data sorts
main = agent.root.b.main
color = agent.root.d.color   
shape = agent.root.d.shape  
number = agent.root.d.number 

# Define three chunks. Technically, these chunks are represented as terms.
# Note that dimension-value pairs are constructed by pairing together two terms 
# using the '**' operator. Positive sign indicates a top-down weight of +1.0. 
# Other weights may be assigned by left multiplication with floats.
chunk_defs = [
    # The caret operator can be used to name chunks and rules. 
    "one_blue_triangle" ^ 
    + main.input ** color.blu
    + main.input ** shape.tria
    + main.input ** number.one,

    "one_red_triangle" ^
    + main.input ** color.red
    + main.input ** shape.tria
    + main.input ** number.one,

    "two_green_squares" ^
    + main.input ** color.grn
    + main.input ** shape.squr
    + main.input ** number.two,
]

# Schedule an event to populate the model with the new chunks. This event will 
# bind the terms defined above to the model's keyspace and initialize their 
# top-down and bottom-up weights. The knowledge defined in `chunk_defs` will not 
# be available to the model until this event is executed (during event 
# processing).
agent.system.schedule(agent.chunks.encode(*chunk_defs))

### Event Processing

Event processing is the process of getting a model to generate and process simulation events. 

Events have four primary effects on the state of a simulation: They may (i) update numerical data in process sites, (ii) update the simulation's keyspace, (iii) cause processes to schedule further events, or (iv) advance the simulation clock. Events do not actively perform computations, thus updates may only depend on the state of a model at the time that an event is scheduled.

Scheduled events are maintained in an event queue, which ensures that they are processed in due time. The preceding step has already generated one event which, when processed, will have all of the effects listed above.  


In [6]:
# List all currently scheduled events. Technically, the event queue is 
# implemented as a heap, so events are not guaranteed to be listed in order.
agent.system.queue

[<Event source=agent.chunks.encode time=datetime.timedelta(0) at 0x1079ea440>]

To facilitate tracking events and other relevant data during the course of a simulation, event logs may be generated using Python's `logging` module.

In [7]:
"""Initialize event logging for simulation"""

import logging
import sys

logger = logging.getLogger("pyClarion.events.system")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.addHandler(logging.StreamHandler(sys.stdout))

The present simulation may be completed simply by sending some inputs into the bottom level of the model.

In [8]:
"""Run simulation"""

# Schedule an event to update the input to the model with the following data.
# Positives get +1.0 activation, negatives get -1.0 activation. 
agent.system.schedule(
    agent.ipt.send(    
        + main.input ** color.blu
        + main.input ** shape.tria
        + main.input ** number.one
        - main.input ** number.two
    ) 
)

# The following lines process all events in the current queue.
print("Event summaries generated by logger.\n")
for event in agent.run(): # Process and yield the next event.
    continue
    # Can respond to event here (e.g., record data, stop simulation, 
    # communicate with external resources etc.)

Event summaries generated by logger.

event 0x0000 00:00:00.00 096 0 agent.chunks.encode
    Added the following new chunk(s)
    chunk d:agent.chunks:one_blue_triangle
        + main.input ** color.blu
        + main.input ** shape.tria
        + main.input ** number.one
    chunk d:agent.chunks:one_red_triangle
        + main.input ** color.red
        + main.input ** shape.tria
        + main.input ** number.one
    chunk d:agent.chunks:two_green_squares
        + main.input ** color.grn
        + main.input ** shape.squr
        + main.input ** number.two
event 0x0000 00:00:00.00 096 2 agent.chunks.encode_weights
event 0x0000 00:00:00.00 064 1 agent.ipt.send
event 0x0000 00:00:00.00 064 3 agent.bu.forward


## Results

The result of computing bottom-up activations from the given input may be obtained by accessing the main output site of the bottom-up activation process `chunks.bu`. This presents an occasion for a glimpse into the internal representations of a `pyClarion` model.

In [9]:
# The bottom-up output is stored in the data site called bu.main. The 
# current data at a site is accessed via subscripting at index 0. Some sites 
# retain lagged data, which may be accessed by subscripting with the 
# corresponding lag value.
data = agent.bu.main[0]

# Print the outcome of the bottom-up activation process.
# The first line of the result contains some general information, each 
# subsequent line lists key-value associations which, in this case, represent 
# bottom-up activations for each chunk. Consult the event logs for information 
# on the chunk identifiers.
print(data)

NumDict 'd:agent.chunks:?' c=0.0
    d:agent.chunks:one_blue_triangle 0.75
    d:agent.chunks:one_red_triangle  0.5
    d:agent.chunks:two_green_squares -0.25


## Conclusion

The `pyClarion` library presents an integrated suite of tools for implementing cognitive models within the Clarion cognitive architecture (and even beyond). It spans a wide range of computational domains including discrete event simulation, knowledge representation, and numerical computation. 

This tutorial introduces some of the main features and workflows of these tools. Specifically, it illustrates the four main activities involved in model implementation: keyspace definition, model construction, knowledge initialization, and event processing. As such, the tutorial provides preparation for more advanced topics including detailed treatments of knowledge representation, neural networks, and process customization. 

Finally, it is worth highlighting that this tutorial focused on implementation steps assuming that the model specification is known in advance. The process of developing novel models typically involves additional intellectual work as specifications are generally not know in advance, at least not in complete detail. In such cases, it is usually beneficial to consider each implementation step in reverse order (i.e., starting out with the desired behavior, translating it to (tentative) event processing specifications, etc.) before starting the implementation.